# SIH Weather-Station Quality System — Scientific Improvement Notebook

This Colab template upgrades a Phase 10 weather-station anomaly, diagnosis, and correction pipeline while preserving an untouched baseline. It is deliberately leakage-safe and event-oriented. Replace only the schema/config values marked `TODO`; do not tune on either final test set.

**Primary operational objective:** maximize early episode recovery under a fixed false-alarm budget. Point accuracy is not a model-selection metric.

**Partition order:** train → tune → calibration → frozen unseen-time test; unseen stations remain completely held out.

## 0. Colab runtime
Use a GPU runtime only for the optional TCN. CatBoost/LightGBM and the rule/specialist system should also be benchmarked on CPU because deployment throughput cannot be inferred from Colab GPU speed.

In [ ]:
!pip -q install catboost lightgbm optuna ruptures psutil codecarbon shap pyarrow

import os, json, time, random, hashlib, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    brier_score_loss, log_loss)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier, CatBoostRegressor
import psutil

SEEDS = [17, 29, 41, 53, 67]
SEED = SEEDS[0]
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings('ignore')
print({'python': platform.python_version(), 'cpu_count': os.cpu_count(),
       'ram_gb': round(psutil.virtual_memory().total/2**30, 2)})

## 1. Configuration and immutable data contract
Edit this cell once. Prefer an existing `split` column produced by your Phase 10 manifest. Never silently fall back to a random split.

In [ ]:
CFG = {
    'data_path': '/content/weather_quality.parquet',  # TODO
    'timestamp': 'timestamp',
    'station': 'station_id',
    'sensors': ['temperature_c', 'pressure_hpa', 'humidity_pct'],
    'anomaly_label': 'is_anomaly',
    'fault_label': 'fault_type',
    'weather_label': 'is_genuine_weather',
    'episode_id': 'episode_id',
    'split_col': 'split',
    'clean_targets': {
        'temperature_c': 'clean_temperature_c',
        'pressure_hpa': 'clean_pressure_hpa',
        'humidity_pct': 'clean_humidity_pct'},
    'expected_cadence_min': 15,  # TODO
    'merge_gap_points': 1,
    'min_event_points': 1,
    'false_alarm_budget': 0.02,
    'min_point_precision': 0.75,
    'target_point_recall': 0.65,
    'target_median_delay_min': 60.0,
    'artifact_dir': '/content/sih_quality_artifacts'
}
Path(CFG['artifact_dir']).mkdir(parents=True, exist_ok=True)
VALID_SPLITS = {'train','tune','calibration','test_time','test_station'}
CFG

In [ ]:
def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def load_frame(path):
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f'Upload/mount dataset: {p}')
    if p.suffix == '.parquet': return pd.read_parquet(p)
    if p.suffix == '.csv': return pd.read_csv(p)
    raise ValueError('Use parquet or csv')

df = load_frame(CFG['data_path'])
required = [CFG['timestamp'], CFG['station'], *CFG['sensors'],
            CFG['anomaly_label'], CFG['fault_label'], CFG['weather_label'], CFG['split_col']]
missing = sorted(set(required) - set(df.columns))
assert not missing, f'Missing required columns: {missing}'
df[CFG['timestamp']] = pd.to_datetime(df[CFG['timestamp']], utc=True)
df = df.sort_values([CFG['station'], CFG['timestamp']]).reset_index(drop=True)
assert set(df[CFG['split_col']].dropna().unique()) <= VALID_SPLITS
assert {'test_time','test_station'} <= set(df[CFG['split_col']].unique())
print('rows:', len(df), 'stations:', df[CFG['station']].nunique(), 'sha256:', sha256_file(CFG['data_path']))

## 2. Leakage and integrity audit
Unseen-station IDs must not appear in train/tune/calibration. Confirm timestamps are unique per station, labels are internally consistent, and clean correction targets are never used as detector inputs.

In [ ]:
S, T, Y, SPL = CFG['station'], CFG['timestamp'], CFG['anomaly_label'], CFG['split_col']
unseen = set(df.loc[df[SPL].eq('test_station'), S].unique())
development = set(df.loc[df[SPL].isin(['train','tune','calibration']), S].unique())
assert unseen.isdisjoint(development), 'LEAKAGE: unseen station occurs in development partitions'
dupes = df.duplicated([S, T], keep=False)
audit = {
    'rows': len(df), 'stations': df[S].nunique(), 'duplicate_station_timestamps': int(dupes.sum()),
    'missing_fraction': df[CFG['sensors']].isna().mean().to_dict(),
    'split_rows': df[SPL].value_counts().to_dict(),
    'split_positive_rate': df.groupby(SPL)[Y].mean().to_dict(),
    'unseen_station_count': len(unseen)
}
display(pd.crosstab(df[SPL], df[CFG['fault_label']], margins=True))
display(pd.json_normalize(audit, sep='.').T.rename(columns={0:'value'}))
Path(CFG['artifact_dir'], 'audit.json').write_text(json.dumps(audit, indent=2, default=str))

In [ ]:
# Freeze a manifest before modelling. Store it with every result.
manifest = (df.groupby(SPL).agg(rows=(S,'size'), stations=(S,'nunique'),
                                  start=(T,'min'), end=(T,'max'), positives=(Y,'sum'))
              .reset_index())
display(manifest)
manifest.to_csv(Path(CFG['artifact_dir'], 'split_manifest.csv'), index=False)

## 3. Causal feature engineering
Every rolling statistic is shifted by one sample before the current observation is scored. Add operational neighbour features only when they truly exist at inference time. Do not include station ID for the unseen-station model.

In [ ]:
def rolling_slope(a):
    a = np.asarray(a, float)
    ok = np.isfinite(a)
    if ok.sum() < 3: return np.nan
    x = np.arange(len(a))[ok]
    return np.polyfit(x, a[ok], 1)[0]

def add_causal_features(frame, cfg):
    z = frame.copy().sort_values([cfg['station'], cfg['timestamp']])
    ts = z[cfg['timestamp']]
    z['hour_sin'] = np.sin(2*np.pi*ts.dt.hour/24); z['hour_cos'] = np.cos(2*np.pi*ts.dt.hour/24)
    z['doy_sin'] = np.sin(2*np.pi*ts.dt.dayofyear/365.25); z['doy_cos'] = np.cos(2*np.pi*ts.dt.dayofyear/365.25)
    g = z.groupby(cfg['station'], sort=False)
    for c in cfg['sensors']:
        z[f'{c}_lag1'] = g[c].shift(1)
        z[f'{c}_diff1'] = z[c] - z[f'{c}_lag1']
        for w in (4, 12, 24, 96):
            prior = g[c].shift(1)
            med = prior.groupby(z[cfg['station']]).transform(lambda s: s.rolling(w, min_periods=max(3,w//4)).median())
            mad = (prior-med).abs().groupby(z[cfg['station']]).transform(lambda s: s.rolling(w, min_periods=max(3,w//4)).median())
            z[f'{c}_med_{w}'] = med
            z[f'{c}_rz_{w}'] = (z[c]-med)/(1.4826*mad+1e-6)
        z[f'{c}_roll_std_12'] = prior.groupby(z[cfg['station']]).transform(lambda s: s.rolling(12,min_periods=4).std())
        z[f'{c}_slope_24'] = prior.groupby(z[cfg['station']]).transform(lambda s: s.rolling(24,min_periods=8).apply(rolling_slope, raw=True))
        z[f'{c}_same_as_prev'] = z[c].eq(z[f'{c}_lag1']).astype('int8')
    z['dewpoint_proxy'] = z['temperature_c'] - (100-z['humidity_pct'])/5 if {'temperature_c','humidity_pct'} <= set(z) else np.nan
    return z

feat = add_causal_features(df, CFG)
for clean_col in CFG['clean_targets'].values():
    assert clean_col not in [c for c in feat.columns if c.endswith(('_lag1','_rz_4','_rz_12','_rz_24','_rz_96'))]
print('feature columns:', len(feat.columns))

## 4. Deterministic rule channel and specialist scores
Adapt physical bounds/rates to the problem owner's published sensor specifications. Duplicate and timestamp evidence must be evaluated in the unified prediction table—not in a separate demo-only counter.

In [ ]:
def sigmoid_abs(x, centre=3.0, scale=1.0):
    return 1/(1+np.exp(-(np.abs(np.asarray(x))-centre)/scale))

def add_rule_scores(z, cfg):
    x = z.copy()
    S,T = cfg['station'], cfg['timestamp']
    x['rule_duplicate'] = x.duplicated([S,T], keep=False).astype(float)
    dt = x.groupby(S)[T].diff().dt.total_seconds().div(60)
    x['rule_timestamp'] = ((dt <= 0) | (dt > 1.5*cfg['expected_cadence_min'])).astype(float)
    x['rule_physical'] = 0.0
    if 'humidity_pct' in x: x['rule_physical'] = np.maximum(x['rule_physical'], (~x['humidity_pct'].between(0,100)).astype(float))
    if 'pressure_hpa' in x: x['rule_physical'] = np.maximum(x['rule_physical'], (~x['pressure_hpa'].between(800,1100)).astype(float))
    if 'temperature_c' in x: x['rule_physical'] = np.maximum(x['rule_physical'], (~x['temperature_c'].between(-60,60)).astype(float))
    rz = [c for c in x if c.endswith('_rz_24')]
    slopes = [c for c in x if c.endswith('_slope_24')]
    stds = [c for c in x if c.endswith('_roll_std_12')]
    same = [c for c in x if c.endswith('_same_as_prev')]
    x['score_spike_drop'] = np.nanmax(np.column_stack([sigmoid_abs(x[c],3.5,0.8) for c in rz]), axis=1)
    x['score_bias_drift'] = np.nanmax(np.column_stack([sigmoid_abs(x[c],2.5,0.8) for c in slopes]), axis=1)
    x['score_frozen'] = np.nanmax(np.column_stack([x[c].rolling(8,min_periods=4).mean() for c in same]), axis=1)
    x['score_noise'] = np.nanmax(np.column_stack([sigmoid_abs(x[c],2.0,0.7) for c in stds]), axis=1)
    x['rule_any'] = x[['rule_duplicate','rule_timestamp','rule_physical']].max(axis=1)
    return x

feat = add_rule_scores(feat, CFG)
display(feat[['rule_duplicate','rule_timestamp','rule_physical','score_spike_drop','score_bias_drift','score_frozen','score_noise']].describe().T)

## 5. Phase 10 baseline adapter
Load or call the untouched Phase 10 predictor here. Its output must be aligned one-to-one with the rows and saved before any new experiment.

In [ ]:
# TODO: replace with your real Phase 10 inference without retraining/tuning on tests.
# feat['phase10_score'] = phase10_predict_proba(feat)
if 'phase10_score' not in feat:
    feat['phase10_score'] = np.nan
print('Connect Phase 10 adapter before claiming an ablation improvement.')

## 6. Global station-independent detector
Use class weighting rather than rowwise SMOTE. Keep IDs, labels, episode IDs, clean targets, and non-operational future information out of `FEATURES`.

In [ ]:
exclude = {CFG['timestamp'], CFG['station'], CFG['anomaly_label'], CFG['fault_label'],
           CFG['weather_label'], CFG['episode_id'], CFG['split_col'], *CFG['clean_targets'].values()}
FEATURES = [c for c in feat.columns if c not in exclude and pd.api.types.is_numeric_dtype(feat[c])]
# Prevent deterministic rules from being learned as soft features; fuse them explicitly later.
MODEL_FEATURES = [c for c in FEATURES if not c.startswith('rule_') and c != 'phase10_score']
tr = feat[SPL].eq('train'); tu = feat[SPL].eq('tune'); ca = feat[SPL].eq('calibration')
pos = feat.loc[tr,Y].sum(); neg = tr.sum()-pos
class_weight = float(neg/max(pos,1))
detector = CatBoostClassifier(iterations=800, depth=7, learning_rate=0.04,
    loss_function='Logloss', eval_metric='PRAUC', scale_pos_weight=class_weight,
    random_seed=SEED, verbose=100, od_type='Iter', od_wait=80,
    task_type='GPU' if os.environ.get('COLAB_GPU') else 'CPU')
detector.fit(feat.loc[tr,MODEL_FEATURES], feat.loc[tr,Y],
             eval_set=(feat.loc[tu,MODEL_FEATURES], feat.loc[tu,Y]))
feat['ml_raw'] = detector.predict_proba(feat[MODEL_FEATURES])[:,1]
print('Tune AUPRC:', average_precision_score(feat.loc[tu,Y], feat.loc[tu,'ml_raw']))

## 7. Optional GPU causal TCN ablation
Build fixed-length windows separately within each station, never across a station boundary. The final timestep label is the target. Use only past/current operational features and chronological sampling. Keep the network small enough for edge deployment.

In [ ]:
import torch
import torch.nn as nn

class CausalBlock(nn.Module):
    def __init__(self, channels, dilation, kernel=3, dropout=0.1):
        super().__init__(); pad=(kernel-1)*dilation
        self.pad=pad
        self.net=nn.Sequential(nn.Conv1d(channels,channels,kernel,padding=pad,dilation=dilation),
                               nn.ReLU(),nn.Dropout(dropout),
                               nn.Conv1d(channels,channels,kernel,padding=pad,dilation=dilation),nn.ReLU())
    def forward(self,x):
        y=self.net(x)
        if self.pad: y=y[...,:-2*self.pad] if y.shape[-1]-2*self.pad==x.shape[-1] else y[...,:x.shape[-1]]
        return x+y

class SmallTCN(nn.Module):
    def __init__(self,n_features,channels=64):
        super().__init__(); self.inp=nn.Conv1d(n_features,channels,1)
        self.blocks=nn.ModuleList([CausalBlock(channels,d) for d in (1,2,4,8,16)])
        self.head=nn.Linear(channels,1)
    def forward(self,x):
        z=self.inp(x.transpose(1,2))
        for b in self.blocks: z=b(z)
        return self.head(z[:,:,-1]).squeeze(-1)

print('TCN receptive field must be checked against cadence and longest drift/bias episode.')
# TODO: implement a station-safe Dataset, weighted episode sampler, early stopping on tune AUPRC,
# and export `tcn_raw`. Do not proceed if window construction crosses station/split boundaries.

## 8. Calibration and hybrid fusion
Split calibration chronologically into three parts: A1 fits score calibration, A2 fits fusion, and B selects the operating policy. This prevents threshold overfitting and avoids fitting fusion on the same rows used to fit the score calibrator.

In [ ]:
cal_idx = feat.index[ca].to_numpy()
cal_idx = cal_idx[np.argsort(feat.loc[cal_idx,T].to_numpy())]
cut1, cut2 = len(cal_idx)//3, 2*len(cal_idx)//3
cal_a1, cal_a2, cal_b = cal_idx[:cut1], cal_idx[cut1:cut2], cal_idx[cut2:]
assert min(len(cal_a1),len(cal_a2),len(cal_b))>0, 'Calibration partition too small for three-way split'
assert feat.loc[cal_a1,Y].nunique()==2 and feat.loc[cal_a2,Y].nunique()==2, 'Each calibration-fit block needs both classes'
iso = IsotonicRegression(out_of_bounds='clip').fit(feat.loc[cal_a1,'ml_raw'], feat.loc[cal_a1,Y])
feat['ml_cal'] = iso.transform(feat['ml_raw'])
fusion_cols = ['ml_cal','score_spike_drop','score_bias_drift','score_frozen','score_noise']
if feat['phase10_score'].notna().all(): fusion_cols.append('phase10_score')
fusion = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED)
fusion.fit(feat.loc[cal_a2,fusion_cols].fillna(0), feat.loc[cal_a2,Y])
feat['fusion_soft'] = fusion.predict_proba(feat[fusion_cols].fillna(0))[:,1]
# Deterministic communication/physics faults override the soft probability.
feat['hybrid_score'] = np.maximum(feat['fusion_soft'], feat['rule_any'])
print('Calibration-B AUPRC:', average_precision_score(feat.loc[cal_b,Y], feat.loc[cal_b,'hybrid_score']))

## 9. Strict event evaluation and operating-point selection
False alarms are defined here as predicted episodes that overlap no true episode, divided by observed station-days. Record this definition beside every result.

In [ ]:
def mark_runs(y, merge_gap=0):
    y=np.asarray(y,bool).copy()
    if merge_gap>0:
        on=np.flatnonzero(y)
        for a,b in zip(on[:-1],on[1:]):
            if 0 < b-a-1 <= merge_gap: y[a:b+1]=True
    starts=np.flatnonzero(y & ~np.r_[False,y[:-1]])
    ends=np.flatnonzero(y & ~np.r_[y[1:],False])
    return list(zip(starts,ends))

def apply_hysteresis(score, start_thr, continue_thr):
    out=np.zeros(len(score),dtype=bool); active=False
    for i,s in enumerate(np.nan_to_num(score,nan=0.0)):
        if not active and s>=start_thr: active=True
        elif active and s<continue_thr: active=False
        out[i]=active
    return out

def event_metrics(frame, pred_col, cfg):
    tp_e=fp_e=fn_e=0; delays=[]; station_days=0.0
    for _,g in frame.groupby(cfg['station'],sort=False):
        g=g.sort_values(cfg['timestamp']); yt=g[cfg['anomaly_label']].to_numpy(bool); yp=g[pred_col].to_numpy(bool)
        true_e=mark_runs(yt,cfg['merge_gap_points']); pred_e=mark_runs(yp,cfg['merge_gap_points'])
        station_days += max((g[cfg['timestamp']].iloc[-1]-g[cfg['timestamp']].iloc[0]).total_seconds()/86400, 1/1440)
        hit_true=set(); hit_pred=set()
        for i,(a,b) in enumerate(true_e):
            overlaps=[(j,c,d) for j,(c,d) in enumerate(pred_e) if j not in hit_pred and c<=b and d>=a]
            if overlaps:
                j,c,d=overlaps[0]; hit_true.add(i); hit_pred.add(j)
                first=max(a,c); delays.append(max(0,first-a)*cfg['expected_cadence_min'])
        tp_e += len(hit_true); fn_e += len(true_e)-len(hit_true); fp_e += len(pred_e)-len(hit_pred)
    ep=tp_e/max(tp_e+fp_e,1); er=tp_e/max(tp_e+fn_e,1); ef1=2*ep*er/max(ep+er,1e-12)
    return {'event_precision':ep,'event_recall':er,'event_f1':ef1,
            'false_alarm_episodes_per_station_day':fp_e/max(station_days,1e-12),
            'delay_median_min':float(np.median(delays)) if delays else np.inf,
            'delay_p90_min':float(np.quantile(delays,.9)) if delays else np.inf,
            'delay_mean_min':float(np.mean(delays)) if delays else np.inf}

def point_metrics(y,p,score):
    return {'auprc':average_precision_score(y,score), 'precision':precision_score(y,p,zero_division=0),
            'recall':recall_score(y,p,zero_division=0), 'f1':f1_score(y,p,zero_division=0)}

def choose_policy(frame, idx, score_col, cfg):
    rows=[]
    for hi in np.linspace(.10,.99,90):
        for gap in (.00,.05,.10,.15):
            lo=max(0,hi-gap); part=frame.loc[idx].copy()
            part['pred']=False
            for _,ix in part.groupby(cfg['station'],sort=False).groups.items():
                part.loc[ix,'pred']=apply_hysteresis(part.loc[ix,score_col].to_numpy(),hi,lo)
            m={**point_metrics(part[cfg['anomaly_label']],part['pred'],part[score_col]), **event_metrics(part,'pred',cfg)}
            m.update(start_threshold=hi,continue_threshold=lo)
            m['event_f2']=5*m['event_precision']*m['event_recall']/max(4*m['event_precision']+m['event_recall'],1e-12)
            rows.append(m)
    tab=pd.DataFrame(rows)
    feasible=tab[(tab.precision>=cfg['min_point_precision']) &
                 (tab.false_alarm_episodes_per_station_day<=cfg['false_alarm_budget'])]
    if len(feasible): return feasible.sort_values(['event_f2','delay_median_min'],ascending=[False,True]).iloc[0],tab
    tab['violation']=np.maximum(0,cfg['min_point_precision']-tab.precision)/cfg['min_point_precision'] + np.maximum(0,tab.false_alarm_episodes_per_station_day-cfg['false_alarm_budget'])/cfg['false_alarm_budget']
    return tab.sort_values(['violation','event_f2'],ascending=[True,False]).iloc[0],tab

policy, frontier = choose_policy(feat, cal_b, 'hybrid_score', CFG)
display(policy.to_frame('selected'))
frontier.to_csv(Path(CFG['artifact_dir'],'calibration_threshold_frontier.csv'),index=False)

## 10. Freeze once, then evaluate both final tests
Run this cell only after features, fusion, and policy are fixed. Never revise the system from test results. Create a new development cycle instead.

In [ ]:
def score_split(frame, split_name, score_col, policy, cfg):
    part=frame.loc[frame[cfg['split_col']].eq(split_name)].copy(); part['pred']=False
    for _,ix in part.groupby(cfg['station'],sort=False).groups.items():
        part.loc[ix,'pred']=apply_hysteresis(part.loc[ix,score_col].to_numpy(),policy.start_threshold,policy.continue_threshold)
    return {**{'split':split_name}, **point_metrics(part[cfg['anomaly_label']],part['pred'],part[score_col]),
            **event_metrics(part,'pred',cfg)}, part

final_rows=[]; final_predictions={}
for split_name in ('test_time','test_station'):
    m,pred=score_split(feat,split_name,'hybrid_score',policy,CFG)
    final_rows.append(m); final_predictions[split_name]=pred
final=pd.DataFrame(final_rows); display(final)
final.to_csv(Path(CFG['artifact_dir'],'final_detection_metrics.csv'),index=False)
pd.concat(final_predictions.values()).to_parquet(Path(CFG['artifact_dir'],'final_predictions.parquet'),index=False)

## 11. Station/block bootstrap confidence intervals
Resample stations, and within a station optionally resample whole days/episodes—not individual rows. Use the same resampled units for paired Phase 10 versus candidate comparisons.

In [ ]:
def station_bootstrap(frame, metric_fn, n_boot=1000, seed=17):
    rng=np.random.default_rng(seed); stations=frame[S].unique(); vals=[]
    for _ in range(n_boot):
        sampled=rng.choice(stations,size=len(stations),replace=True)
        blocks=[]
        for k,s in enumerate(sampled):
            b=frame.loc[frame[S].eq(s)].copy(); b[S]=f'{s}__boot{k}'; blocks.append(b)
        vals.append(metric_fn(pd.concat(blocks,ignore_index=True)))
    return np.quantile(vals,[.025,.5,.975])

for name,pred in final_predictions.items():
    ci=station_bootstrap(pred,lambda q:event_metrics(q,'pred',CFG)['event_recall'],n_boot=500)
    print(name,'event-recall 95% CI/median:',ci)

## 12. Per-fault failure analysis
Episode-level recall—not row recall—must drive frozen, bias, drift, and duplicate improvements. Save examples around the earliest miss/false alarm for review.

In [ ]:
def per_fault_event_recall(frame, cfg):
    rows=[]
    for fault,g in frame.loc[frame[cfg['anomaly_label']].eq(1)].groupby(cfg['fault_label']):
        episode_col=cfg['episode_id'] if cfg['episode_id'] in g else None
        if episode_col:
            hit=g.groupby(episode_col)['pred'].any(); recall=hit.mean(); n=len(hit)
        else:
            recall=np.nan; n=0
        rows.append({'fault':fault,'episode_recall':recall,'episodes':n})
    return pd.DataFrame(rows).sort_values('episode_recall')

for name,pred in final_predictions.items():
    print(name); display(per_fault_event_recall(pred,CFG))

## 13. Hierarchical weather/source and root-cause diagnosis
Detect first; then distinguish genuine weather, sensor/communication fault, both/uncertain, or normal. Evaluate diagnosis both with oracle detections and end-to-end predicted detections. Plot accepted accuracy versus coverage and choose an abstention threshold on calibration only.

In [ ]:
diag_train=feat.loc[tr & feat[Y].eq(1)].copy(); diag_tune=feat.loc[tu & feat[Y].eq(1)].copy()
le=LabelEncoder().fit(diag_train[CFG['fault_label']].astype(str))
known_tune=diag_tune[CFG['fault_label']].astype(str).isin(le.classes_)
diag_tune=diag_tune.loc[known_tune]
diag=CatBoostClassifier(iterations=600,depth=7,learning_rate=.04,loss_function='MultiClass',
    auto_class_weights='Balanced',random_seed=SEED,verbose=100)
diag.fit(diag_train[MODEL_FEATURES],le.transform(diag_train[CFG['fault_label']].astype(str)),
         eval_set=(diag_tune[MODEL_FEATURES],le.transform(diag_tune[CFG['fault_label']].astype(str))))
# TODO: apply deterministic overrides for duplicate/timestamp/physical faults, calibrate class probabilities,
# and choose the confidence threshold using calibration risk-coverage curves.
def risk_coverage(proba,y_true):
    conf=proba.max(1); pred=proba.argmax(1); rows=[]
    for th in np.linspace(.3,.99,70):
        keep=conf>=th
        rows.append({'threshold':th,'coverage':keep.mean(),
                     'accepted_accuracy':(pred[keep]==y_true[keep]).mean() if keep.any() else np.nan})
    return pd.DataFrame(rows)

## 14. Correction models and adaptive conformal intervals
Train only on normal observations with clean/verified targets. Compare learned correction against persistence and seasonal-median baselines. Never overwrite the raw value. Calibrate intervals per sensor and report coverage plus width on anomalous rows.

In [ ]:
correction_models={}; conformal_q={}
for sensor,target in CFG['clean_targets'].items():
    if target not in feat: print('skip missing clean target:',target); continue
    use_train=tr & feat[target].notna() & feat[Y].eq(0)
    use_tune=tu & feat[target].notna()
    reg=CatBoostRegressor(iterations=700,depth=7,learning_rate=.04,loss_function='MAE',
                          random_seed=SEED,verbose=False)
    reg.fit(feat.loc[use_train,MODEL_FEATURES],feat.loc[use_train,target],
            eval_set=(feat.loc[use_tune,MODEL_FEATURES],feat.loc[use_tune,target]),verbose=100)
    feat[f'{sensor}_corrected']=reg.predict(feat[MODEL_FEATURES])
    cal_res=(feat.loc[ca & feat[target].notna(),target]-feat.loc[ca & feat[target].notna(),f'{sensor}_corrected']).abs()
    q=float(np.quantile(cal_res,.90,method='higher'))
    feat[f'{sensor}_lo90']=feat[f'{sensor}_corrected']-q; feat[f'{sensor}_hi90']=feat[f'{sensor}_corrected']+q
    correction_models[sensor]=reg; conformal_q[sensor]=q

rows=[]
for split_name in ('test_time','test_station'):
  for sensor,target in CFG['clean_targets'].items():
    if f'{sensor}_corrected' not in feat or target not in feat: continue
    m=feat[SPL].eq(split_name)&feat[Y].eq(1)&feat[target].notna()
    err=(feat.loc[m,f'{sensor}_corrected']-feat.loc[m,target]).abs()
    cover=((feat.loc[m,target]>=feat.loc[m,f'{sensor}_lo90'])&(feat.loc[m,target]<=feat.loc[m,f'{sensor}_hi90'])).mean()
    rows.append({'split':split_name,'sensor':sensor,'mae':err.mean(),'interval_coverage_90':cover,
                 'mean_width':(feat.loc[m,f'{sensor}_hi90']-feat.loc[m,f'{sensor}_lo90']).mean(),'n':int(m.sum())})
display(pd.DataFrame(rows))
print('For nonstationary residuals, replace the fixed quantile with rolling/adaptive conformal updates using only revealed past clean residuals.')

## 15. Ablation ledger
Required rows: Phase 10, rules only, ML only, rules+residual, +specialists, +calibration/fusion, +hysteresis, +weather hierarchy, +TCN. Evaluate every candidate with the same frozen splits and definitions.

In [ ]:
ABLATION_COLUMNS=['variant','seed','split','auprc','precision','recall','f1','event_precision','event_recall','event_f1',
 'false_alarm_episodes_per_station_day','delay_median_min','delay_p90_min','latency_ms_per_row','model_mb','peak_ram_mb']
ablation=pd.DataFrame(columns=ABLATION_COLUMNS)
# TODO: append each controlled variant and all five seeds. Never report only the winning seed.
ablation.to_csv(Path(CFG['artifact_dir'],'ablation_ledger.csv'),index=False)
ablation

## 16. End-to-end resource benchmark
Measure parse → feature generation → model → fusion → post-processing → JSON serialization. Repeat on CPU and target edge hardware. CodeCarbon is an estimate; record hardware, region, duration, and tool version.

In [ ]:
def benchmark_callable(fn, batch, repeats=20):
    fn(batch); times=[]; proc=psutil.Process(os.getpid()); before=proc.memory_info().rss
    for _ in range(repeats):
        t=time.perf_counter(); fn(batch); times.append(time.perf_counter()-t)
    after=proc.memory_info().rss
    return {'rows':len(batch),'median_ms_per_batch':1000*np.median(times),
            'p95_ms_per_batch':1000*np.quantile(times,.95),
            'median_ms_per_row':1000*np.median(times)/max(len(batch),1),
            'rss_delta_mb':(after-before)/2**20}

def detector_only(batch): return detector.predict_proba(batch[MODEL_FEATURES])[:,1]
sample=feat.loc[feat[SPL].eq('test_time')].head(4096)
display(benchmark_callable(detector_only,sample))
# TODO: benchmark the complete API function and serialize all models to measure total size.

## 17. Unified dashboard incident payload
One schema must carry deterministic and ML evidence together. Raw observations remain immutable.

In [ ]:
incident_example={
 'incident_id':'station_timestamp_hash','station_id':'ST001','started_at':'ISO-8601','updated_at':'ISO-8601',
 'status':'alert|review|resolved','raw_values':{},'anomaly_probability':0.0,
 'evidence':{'rules_fired':[],'specialist_scores':{},'top_model_features':[],'neighbour_agreement':None},
 'source':{'class':'sensor_fault|communication|genuine_weather|both_or_uncertain','confidence':0.0},
 'root_cause':{'class':'unknown','confidence':0.0,'abstained':True},
 'correction':{'allowed':False,'values':{},'interval_90':{},'method':'review_only'},
 'recommended_action':'inspect sensor','versions':{'data':'sha256','model':'git/model hash','policy':'hash'},
 'offline':True,'audit_trail':[]
}
print(json.dumps(incident_example,indent=2))
Path(CFG['artifact_dir'],'incident_schema_example.json').write_text(json.dumps(incident_example,indent=2))

## 18. SIH evidence matrix and stop/go checklist
Do not invent an official numerical SIH score. Attach one verifiable artifact to each published qualitative criterion.

| SIH criterion | Evidence to generate |
|---|---|
| Novelty | Hybrid physics/communication + specialist ML + safe correction; ablation proving each part |
| Complexity | Streaming architecture, unseen-station generalization, calibration, offline sync |
| Clarity/detail | Data card, model card, architecture, metric definitions, split manifest |
| Feasibility/practicability | Live demo, end-to-end API latency, failure modes, offline fallback |
| Sustainability | Model size, CPU/RAM/energy estimate, retraining frequency, maintainability |
| Scale of impact | Stations/day supported, avoided bad observations, stakeholder workflow |
| User experience | Incident timeline, evidence, uncertainty, action, operator usability test |
| Future progression | Pilot plan, sensor expansion, monitoring/drift plan, integration API |

### Stop/go rules
- Stop if unseen stations leak into any development transform.
- Stop if final thresholds were selected on a test set.
- Stop if duplicate/timestamp rules are absent from unified evaluation.
- Keep humidity review-only until interval calibration is acceptable across both tests.
- Promote the TCN only if paired station-bootstrap results improve the operational objective and resource limits remain acceptable.